## Climsim Spectral Analysis

In [5]:
import diffusionsim.training_utils as tru
import diffusionsim.mydatasets as data
from pysteps.utils import spectral
import scipy
import scipy.interpolate

In [19]:
dconfig = tru.my_dconfig(data_vars='v2', in_notebook=True, climsim_training=True, batch_size=128)
dconfig.train_test_split = [1.0]
dconfig.dataset_type = "climsim_image"

In [20]:
dsets, indices = data.load_dataset(dconfig, log=False)

In [21]:
ds = dsets[0]

In [27]:
x = ds.X.isel(time=0)

In [34]:
dconfig.dataloader_params.batch_size

49152

In [33]:
ds.xgen

In [17]:
x.shape, y.shape, x.dtype, x.device

(torch.Size([49152, 557]),
 torch.Size([49152, 368]),
 torch.float32,
 device(type='cpu'))

In [18]:
tmap = y[23, 59, :, :]

IndexError: too many indices for tensor of dimension 2

In [ ]:
def fourier(image):
    spec = np.fft.fft2(image)
    spec = np.fft.fftshift(spec) 
    return(np.log(np.abs(spec)), np.angle(spec))

In [ ]:
#import scipy.ndimage
#upsample_factor = 4  # You can adjust this to achieve the desired resolution
#spec_upsampled = scipy.ndimage.zoom(np.abs(spec), upsample_factor)


In [ ]:
mag, phase = fourier(tmap)
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(mag, vmin=-5, vmax=11, cmap='viridis')
plt.colorbar()
plt.subplot(1, 2, 2)
plt.imshow(phase, vmin=-1*np.pi, vmax=np.pi, cmap=plt.cm.RdBu)
plt.colorbar()

In [ ]:
def RAPSD(image, num_slices=60, num_interp_points=100):
    """
    Computes the Radial Average Power Spectral Density for an input image.
    """
    spec = np.fft.fftshift(np.fft.fft2(image))
    
    # Initialize an array to accumulate power across radial slices
    radial_power = np.zeros(num_interp_points)

    # Loop over angles to take radial slices
    for i in range(num_slices):
        # Angle in radians
        angle = i * (2 * np.pi / num_slices)
        x = np.cos(angle)
        y = np.sin(angle)
        
        # Start and end coordinates for the radial line
        x_start = spec.shape[0] / 2
        y_start = spec.shape[1] / 2
        x_end = x_start + x * (spec.shape[0] / 2)
        y_end = y_start + y * (spec.shape[1] / 2)
        
        # Interpolator to sample the FFT spectrum along the radial line
        interp = scipy.interpolate.RegularGridInterpolator(
            (np.arange(spec.shape[0]), np.arange(spec.shape[1])), np.abs(spec) ** 2 / spec.size)

        # Generate line coordinates and ensure they are within bounds
        x_line = np.linspace(x_start, x_end, num_interp_points, endpoint=False)
        y_line = np.linspace(y_start, y_end, num_interp_points, endpoint=False)
        
        # Clip coordinates to stay within image bounds
        x_line = np.clip(x_line, 0, spec.shape[0] - 1)
        y_line = np.clip(y_line, 0, spec.shape[1] - 1)
        
        coords_line = np.array([x_line, y_line]).T
        
        # Sample and accumulate the power along this radial line
        spec_line = interp(coords_line)
        radial_power += spec_line

    # Average the accumulated power over the number of slices to get the RAPSD
    rapsd = radial_power / num_slices
    return rapsd

In [ ]:
r = RAPSD(tmap)

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def RAPSDi(images, indices):
    # Define interactive plot function
    def plot_image(index, variable):
        vmap = images[index, variable, :, :]
        plt.plot(np.log(RAPSD(vmap)))
        plt.ylabel("log power")
        plt.xlabel("frequency")
        plt.title(f"Image {indices[index] + 1} - Variable {ds.mlo[variable].item()}")
        plt.show()

    # Create interactive widgets
    n, d = images.shape[:2]
    index_slider = widgets.IntSlider(value=0, min=0, max=n-1, step=1, description='Index:')
    variable_slider = widgets.IntSlider(value=0, min=0, max=d-1, step=1, description='Variable:')
    # Display interactive plot
    interact(plot_image, index=index_slider, variable=variable_slider)

In [ ]:
indices = np.random.randint(low=0, high=210240, size=100)
images = ds.data.sel(time=indices)
#images.load()

In [ ]:
img = (images - ds.Ymean.mean(dim='ncol')) / ds.Ystd.mean(dim='ncol')
img = img.isel(ncol=ds.permute_indices)
img = img.to_stacked_array(new_dim="mlo", sample_dims=("time", "ncol"))
img = img.transpose("time", "mlo", "ncol")
img = img.data.reshape(-1, 128, ds.height, ds.width)

In [ ]:
plt.imshow(img[5, 59])
plt.colorbar()

In [ ]:
RAPSDi(img, indices)

In [ ]:
for k in range(num_displayed_examples):
  rapsd, frequencies = spectral.rapsd(images[k, ..., 1], fft_method=np.fft, return_freq=True)
  plt.subplot(2, 4, 4 + k + 1)
  plt.plot(frequencies[1:], rapsd[1:], c='red', marker='o', markersize=3)  # Chop off the DC component.
  plt.xscale('log')
  plt.yscale('log')
  plt.xlabel('frequency')
  if k == 0:
    plt.ylabel('power')